In [ ]:
import hashlib
import math
from typing import List, Tuple, Any, Dict

# Simuler les imports - vous devrez implémenter ces fonctions
class Lists:
    @staticmethod
    def map(func, lst):
        return list(map(func, lst))
    
    @staticmethod
    def sum(lst):
        return sum(lst)
    
    @staticmethod
    def reverse(lst):
        return list(reversed(lst))
    
    @staticmethod
    def filter(func, lst):
        return list(filter(func, lst))
    
    @staticmethod
    def append(lst1, lst2):
        return lst1 + lst2
    
    @staticmethod
    def partition(pred, lst):
        """Simule lists:partition/2 d'Erlang"""
        sat = []
        not_sat = []
        for item in lst:
            if pred(item):
                sat.append(item)
            else:
                not_sat.append(item)
        return sat, not_sat

class Sets:
    @staticmethod
    def from_list(lst):
        return set(lst)
    
    @staticmethod
    def to_list(s):
        return list(s)
    
    @staticmethod
    def subtract(s1, s2):
        return s1 - s2
    
    @staticmethod
    def add_element(element, s):
        return s | {element}

class CodingList:
    @staticmethod
    def coding_list(n):
        # À implémenter selon votre logique
        return list(range(n))

class Library:
    @staticmethod
    def second(conf, refvec):
        # À implémenter - retourne le second élément
        if isinstance(conf, tuple) and len(conf) > 1:
            return conf[1]
        elif isinstance(conf, list) and len(conf) > 1:
            return conf[1]
        return None
    
    @staticmethod
    def split(conf, refvec):
        # À implémenter - divise une configuration
        mid = len(conf) // 2
        return (conf[:mid], conf[mid:])
    
    @staticmethod
    def is_Terminal(conf, refvec):
        # À implémenter - vérifie si c'est terminal
        return len(conf) <= 1
    
    @staticmethod
    def getInitialConf(refvec):
        # À implémenter - configuration initiale
        return refvec.copy()
    
    @staticmethod
    def displayOfConf(conf, refvec):
        print(f"Configuration: {conf}")
    
    @staticmethod
    def countSetBits(value):
        # Compter les bits à 1
        if isinstance(value, int):
            return bin(value).count('1')
        # Si c'est une liste/tuple, compter les éléments non nuls
        elif isinstance(value, (list, tuple)):
            return sum(1 for x in value if x)
        return 0

# Utiliser les classes
lists = Lists()
sets = Sets()
codinglist = CodingList()
library = Library()

class TestHash:
    @staticmethod
    def h(conf, M, refvec):
        """
        Fonction de hachage similaire à l'original Erlang
        h(Conf,M,Refvec) -> 
            Snd = second(Conf,Refvec),
            Ssnd = integer_to_list(Snd),
            N1 = crypto:hash(md5, Ssnd),
            (binary:decode_unsigned(N1)) rem M
        """
        snd = library.second(conf, refvec)
        
        # Convertir en string (simule integer_to_list)
        if snd is None:
            ssnd = "0"
        else:
            ssnd = str(snd)
        
        # Hacher avec MD5 (simule crypto:hash(md5, Ssnd))
        hash_obj = hashlib.md5(ssnd.encode('utf-8'))
        n1 = hash_obj.digest()
        
        # Décoder en entier non signé (simule binary:decode_unsigned)
        # et prendre le modulo M
        hash_int = int.from_bytes(n1, byteorder='big', signed=False)
        return hash_int % M
    
    @staticmethod
    def allConfiguration(refvec, S, All):
        """
        Génère toutes les configurations possibles
        
        Traduction de :
        allConfiguration(Refvec,S,All)->   
            case S of 
                [Conf|Tl]-> 
                    B = not(is_Terminal(Conf,Refvec)),
                    B2 = (countSetBits(second(Conf,Refvec)) >= math:sqrt(length(Refvec))),
                    if 
                       (B and B2) -> 
                           {Rs1,Rs2} = split(Conf,Refvec),
                           L = [Rs1,Rs2],
                           S1 = append(L,Tl),
                           A1 = append([Conf],All),
                           allConfiguration(Refvec,S1,A1);
                       (not(B) and B2) -> 
                           T1 = append([Conf],All),
                           allConfiguration(Refvec,Tl,T1);
                       not(B2) -> 
                           allConfiguration(Refvec,Tl,All)
                    end;
                []-> All
            end.
        """
        if not S:  # Cas vide : []
            return All
        
        # Cas [Conf|Tl]
        conf = S[0]
        tl = S[1:]
        
        b = not library.is_Terminal(conf, refvec)
        b2 = (library.countSetBits(library.second(conf, refvec)) >= 
              math.sqrt(len(refvec)))
        
        if b and b2:
            # Diviser la configuration
            rs1, rs2 = library.split(conf, refvec)
            
            # Créer les nouvelles listes
            L = [rs1, rs2]
            S1 = lists.append(L, tl)
            A1 = lists.append([conf], All)
            
            # Appel récursif
            return TestHash.allConfiguration(refvec, S1, A1)
        
        elif (not b) and b2:
            # Ajouter à All et continuer
            T1 = lists.append([conf], All)
            return TestHash.allConfiguration(refvec, tl, T1)
        
        else:  # not B2
            # Continuer sans ajouter
            return TestHash.allConfiguration(refvec, tl, All)
    
    @staticmethod
    def foreach(H, T, L, A):
        """
        Helper function pour partitionner les éléments
        
        Traduction de :
        foreach( [H|T],L,A) ->
            {Sat,NotSat} = lists:partition(fun(X) -> X == H end,L),
            A1 = A++[Sat],
            foreach( T,NotSat,A1);
        foreach( [],_,A) -> A .
        """
        if not H and not T:  # Cas []
            return A
        
        if H is not None:
            current_list = [H] + T if T else [H]
        else:
            current_list = T
        
        if not current_list:  # Liste vide
            return A
        
        h = current_list[0]
        t = current_list[1:] if len(current_list) > 1 else []
        
        # Partitionner L selon si les éléments sont égaux à h
        sat, not_sat = lists.partition(lambda x: x == h, L)
        
        # Ajouter les éléments satisfaits à A
        A1 = A + [sat]
        
        # Appel récursif avec le reste
        return TestHash.foreach(None, t, not_sat, A1)
    
    @staticmethod
    def maph(M, refvec, L):
        """
        Applique la fonction h à chaque élément de L
        
        Traduction de :
        maph(M,Refvec,L)-> [h(C,M,Refvec) || C<-L].
        """
        return [TestHash.h(c, M, refvec) for c in L]
    
    @staticmethod
    def numOfConfByMachines(M, refvec, L):
        """
        Compte le nombre de configurations par machine
        
        Traduction de :
        numOfConfByMachines(M,Refvec,L)-> 
            Sq = lists:seq(0,M-1),
            D = maph(M,Refvec,L),
            Li = foreach( Sq,D,[]), 
            [io_lib:write(length(X)) || X <- Li].
        """
        # Générer la séquence 0..M-1
        sq = list(range(M))
        
        # Calculer les hachages
        D = TestHash.maph(M, refvec, L)
        
        # Partitionner par valeur de hachage
        # Note: La fonction foreach originale attend [H|T] comme premier argument
        # Nous devons adapter l'appel
        Li = TestHash.foreach(sq[0] if sq else None, 
                             sq[1:] if len(sq) > 1 else [], 
                             D, 
                             [])
        
        # Retourner les longueurs de chaque groupe
        return [str(len(x)) for x in Li]


# Exemple d'utilisation
if __name__ == "__main__":
    # Initialisation
    N = 4
    refvec = codinglist.coding_list(N)
    
    # Tester la fonction h
    conf = library.getInitialConf(refvec)
    M = 10
    hash_value = TestHash.h(conf, M, refvec)
    print(f"Hash de la configuration {conf}: {hash_value}")
    
    # Tester maph
    configurations = [conf, conf, conf]
    hashes = TestHash.maph(M, refvec, configurations)
    print(f"Hashes: {hashes}")
    
    # Tester numOfConfByMachines
    counts = TestHash.numOfConfByMachines(M, refvec, configurations)
    print(f"Nombre par machine: {counts}")
    
    # Tester allConfiguration
    S = [conf]
    All = []
    all_configs = TestHash.allConfiguration(refvec, S, All)
    print(f"Toutes les configurations: {all_configs}")